In [1]:
!pip install streamlit pyngrok tensorflow pillow joblib torch transformers pytorch-tabnet --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 47.4 MB/s eta 0:00:00


In [2]:
from pyngrok import ngrok
ngrok.set_auth_token("37HyOGn1uvwb511BkxS86iIkXpr_2x7My17vnhKRHHgJwyLyF")

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%%writefile app.py
import streamlit as st
import numpy as np
import joblib
import torch
import tensorflow as tf

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ===================== BASE PATH =====================
BASE_PATH = "/content/drive/MyDrive/Klasifikasi_Lowongan_Pekerjaan"

# ===================== PAGE CONFIG =====================
st.set_page_config(
    page_title="Klasifikasi Lowongan Pekerjaan Asli / Palsu",
    layout="wide"
)

st.title("💼 Klasifikasi Lowongan Pekerjaan Asli / Palsu")
st.write("Deteksi lowongan **ASLI** atau **PALSU** menggunakan model NLP")

# ===================== SIDEBAR =====================
st.sidebar.header("⚙️ Pengaturan Model")

model_choice = st.sidebar.radio(
    "Pilih Model:",
    ["LSTM", "DistilBERT", "BERT"]
)

st.sidebar.markdown("---")
st.sidebar.info(
    """
    **Keterangan Model:**
    - LSTM → ringan & cepat
    - DistilBERT → seimbang
    - BERT → paling akurat
    """
)

# ===================== CACHE MODELS =====================
@st.cache_resource
def load_lstm_model():
    tf.keras.backend.clear_session()
    model = load_model(f"{BASE_PATH}/lstm/lstm.h5", compile=False)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    tokenizer = joblib.load(f"{BASE_PATH}/lstm/tokenizer.pkl")
    return model, tokenizer


@st.cache_resource
def load_distilbert_model():
    model_path = f"{BASE_PATH}/distilbert"
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.eval()
    return tokenizer, model


@st.cache_resource
def load_bert_model():
    model_path = f"{BASE_PATH}/bert"
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.eval()
    return tokenizer, model


# ===================== LOAD MODELS =====================
lstm_model, lstm_tokenizer = load_lstm_model()
distil_tokenizer, distil_model = load_distilbert_model()
bert_tokenizer, bert_model = load_bert_model()

# ===================== MAIN INPUT =====================
st.header(f"🔍 Prediksi dengan {model_choice}")

text_input = st.text_area(
    "Masukkan teks lowongan pekerjaan:",
    height=220
)

# ===================== PREDICTION =====================
if st.button("🚀 Prediksi"):
    if text_input.strip() == "":
        st.warning("Teks tidak boleh kosong")
    else:
        # -------- LSTM --------
        if model_choice == "LSTM":
            seq = lstm_tokenizer.texts_to_sequences([text_input])
            pad_seq = pad_sequences(
                seq, maxlen=200, padding="post", truncating="post"
            )

            prob = lstm_model.predict(pad_seq)[0][0]
            label = "Lowongan PALSU" if prob > 0.5 else "Lowongan ASLI"
            confidence = prob * 100 if prob > 0.5 else (1 - prob) * 100

        # -------- DistilBERT --------
        elif model_choice == "DistilBERT":
            inputs = distil_tokenizer(
                text_input,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=128
            )

            with torch.no_grad():
                outputs = distil_model(**inputs)

            probs = torch.softmax(outputs.logits, dim=1)
            prob_fraud = probs[0][1].item()

            label = "Lowongan PALSU" if prob_fraud > 0.5 else "Lowongan ASLI"
            confidence = prob_fraud * 100 if prob_fraud > 0.5 else (1 - prob_fraud) * 100

        # -------- BERT --------
        else:
            inputs = bert_tokenizer(
                text_input,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=128
            )

            with torch.no_grad():
                outputs = bert_model(**inputs)

            probs = torch.softmax(outputs.logits, dim=1)
            prob_fraud = probs[0][1].item()

            label = "Lowongan PALSU" if prob_fraud > 0.5 else "Lowongan ASLI"
            confidence = prob_fraud * 100 if prob_fraud > 0.5 else (1 - prob_fraud) * 100

        # ===================== OUTPUT =====================
        if "PALSU" in label:
            st.error(f"### 🚨 Hasil Prediksi: **{label}**")
            st.markdown(
                f"<h4 style='color:red;'>📊 Confidence: {confidence:.2f}%</h4>",
                unsafe_allow_html=True
            )
        else:
            st.success(f"### ✅ Hasil Prediksi: **{label}**")
            st.markdown(
                f"<h4 style='color:green;'>📊 Confidence: {confidence:.2f}%</h4>",
                unsafe_allow_html=True
            )

Writing app.py


In [5]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &>/content/logs.txt &

In [6]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print("🔗 Streamlit URL:", public_url)

🔗 Streamlit URL: NgrokTunnel: "https://bolographic-gnarly-korbin.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
!pkill ngrok